# Component interpretation across k = 4, 6 and 8

This read-only notebook completes the preparation for the uniform interpretive review specified in the follow-up protocol. It reads published coefficients and regenerated full reference matrices; it does not fit models, edit labels, or write review decisions. Preparing this notebook does not complete the human review.

For every component ask the same questions: (1) Is there a specific linguistic pattern? (2) Do representative documents support that description? (3) Are contradictions or repeated formulas disclosed? Keep mixed components mixed. Previously inspected examples are development evidence, not a new independent test.


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

current = Path.cwd()
ROOT = next((p for p in [current, *current.parents]
             if (p / "data/processed/model_corpus_v1.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project root or notebooks directory.")

source = ROOT / "data/processed/model_corpus_v1.csv"
followup = json.loads((ROOT / "results/followup_audit_v1/manifest.json").read_text())
if hashlib.sha256(source.read_bytes()).hexdigest() != followup["source_sha256"]:
    raise ValueError("Model corpus differs from the frozen follow-up input.")

ref_dir = ROOT / "results/reference_numeric_v1"
reference_manifest = json.loads((ref_dir / "manifest.json").read_text())
for name, expected in reference_manifest["files_sha256"].items():
    if hashlib.sha256((ref_dir / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Reference file checksum mismatch: {name}")
terms = np.array(json.loads((ref_dir / "vocabulary.json").read_text()))
corpus = pd.read_csv(source, keep_default_na=False)
if not corpus["execution_number"].is_unique:
    raise ValueError("Duplicate execution IDs in corpus.")

membership = pd.read_csv(ROOT / "results/followup_audit_v1/feature_groups.csv")
if membership["term"].duplicated().any() or set(membership["term"]) != set(terms):
    raise ValueError("Feature groups do not partition the vocabulary.")
lookup = {term: i for i, term in enumerate(terms)}
groups = [np.array([lookup[t] for t in part["term"]])
          for _, part in membership.groupby("group_id", sort=True)]
representatives = np.array([min(terms[g]) for g in groups])

H, W = {}, {}
for k in [4, 6, 8]:
    frame = pd.read_csv(ref_dir / f"topic_term_k{k}.csv.gz")
    if frame.columns.tolist() != terms.tolist() or frame.shape != (k, len(terms)):
        raise ValueError(f"Reference dimensions or vocabulary differ at k={k}.")
    H[k] = frame.to_numpy()
    if not np.isfinite(H[k]).all() or (H[k] < 0).any():
        raise ValueError("Invalid term weights.")
    np.testing.assert_allclose(np.linalg.norm(H[k], axis=1), 1, atol=1e-10)
    weights = pd.read_csv(ROOT / f"results/nmf_v2/document_weights_k{k}.csv")
    if len(weights) != followup.get("documents_modeled", 477):
        raise ValueError("Unexpected modeled-document count.")
    if not weights["execution_number"].is_unique:
        raise ValueError("Duplicate modeled execution IDs.")
    W[k] = weights.merge(corpus[["execution_number", "model_text", "model_word_count", "source_url"]],
                         on="execution_number", how="left", validate="one_to_one", indicator=True)
    if not W[k]["_merge"].eq("both").all():
        raise ValueError("Missing model text.")
if any(set(W[k]["execution_number"]) != set(W[6]["execution_number"]) for k in [4, 8]):
    raise ValueError("Modeled IDs differ across k.")
print("Verified: 477 modeled documents; 3,196 features; 3,011 display groups.")


## Leading feature groups

Groups combine features with identical full-corpus TF-IDF columns for display only. Fitting retains all features. Rank by mean member weight rounded to ten decimal places, then alphabetical representative. A word list is evidence to inspect, not a validated interpretation. The number of components is 18 in total, not 18 independent findings.


In [ ]:
def leading_groups(k, topic, n=10):
    if k not in H or not 1 <= topic <= k:
        raise ValueError("Choose k=4/6/8 and a topic number within that k.")
    scores = np.array([H[k][topic - 1, g].mean() for g in groups])
    order = np.lexsort((representatives, -np.round(scores, 10)))[:n]
    return pd.DataFrame({
        "rank": np.arange(1, len(order) + 1),
        "feature_group": [" | ".join(sorted(terms[groups[i]])) for i in order],
        "mean_member_weight": scores[order],
    })

overview = pd.DataFrame([
    {"k": k, "topic": topic,
     "leading_groups": "; ".join(leading_groups(k, topic, 8)["feature_group"])}
    for k in [4, 6, 8] for topic in range(1, k + 1)
])
with pd.option_context("display.max_colwidth", None, "display.max_rows", 20):
    display(overview.set_index(["k", "topic"]))


## Cross-k lexical correspondence

Read each cell as full-vector cosine similarity. This is not a probability or a semantic equivalence test. There is no forced one-to-one match: one component may correspond to multiple components at another k. Comparisons use regenerated reference term vectors, not evidence that every random initialization has the same structure.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, alternative_k in zip(axes, [4, 8]):
    similarity = H[6] @ H[alternative_k].T
    im = ax.imshow(similarity, vmin=0, vmax=1, cmap="Blues", aspect="auto")
    ax.set_xticks(range(alternative_k), labels=range(1, alternative_k + 1))
    ax.set_yticks(range(6), labels=range(1, 7))
    ax.set_xlabel(f"k={alternative_k} component")
    ax.set_ylabel("k=6 component")
    ax.set_title(f"Lexical correspondence: k=6 versus k={alternative_k}")
    for i in range(6):
        for j in range(alternative_k):
            ax.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center",
                    color="white" if similarity[i, j] > 0.55 else "black")
fig.colorbar(im, ax=axes, label="Cosine similarity")
plt.show()


## Read documents before assigning a name

The default shows the three largest positive coefficients among documents of at least 20 words. This is a fixed display rule, not a corpus exclusion. It favors strong examples and cannot establish component-wide purity. Set `minimum_words=0` to include short documents and vary `offset` to inspect lower-ranked texts using the same ranking. Read beyond the first three before completing a review. Record execution IDs and short evidence excerpts when noting support or contradiction.


In [ ]:
def show_component(k, topic, n_examples=3, minimum_words=20, offset=0):
    display(leading_groups(k, topic, 15))
    frame = W[k]
    coefficient = f"topic_{topic}"
    eligible = frame.loc[(frame[coefficient] > 0)
                         & frame["model_word_count"].ge(minimum_words)]
    ranked = eligible.sort_values([coefficient, "execution_number"], ascending=[False, True])
    selected = ranked.iloc[offset:offset + n_examples]
    print(f"k={k}, component={topic}; eligible documents={len(eligible)}")
    for _, row in selected.iterrows():
        print("\n" + "=" * 70)
        print(f"Execution {int(row['execution_number'])}; coefficient={row[coefficient]:.6f}")
        print("Source:", row["source_url"])
        print(row["model_text"])

show_component(6, 1)


## Review notes to complete manually

For each (k, component), record: a narrow provisional description; supporting execution IDs and quotations; contradictory or ambiguous examples; repeated-formula influence; and whether the component remains mixed or unresolved. No suggested human labels are prefilled.

Start with k=6 components 1 and 4: is the first about apology, forgiveness, harm acknowledgment, or a mixture; does the fourth capture repeated prayer rather than broad faith? Then review the remaining components and compare k=4/8 using the same questions. Only after all 18 reviews should the protocol's interpretive part be marked complete. The existing frozen eight-label annotations remain unchanged.
